# 08 — Enhanced Static Benchmark and Threshold Tuning

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This stage improves the static susceptibility experiment without changing the basic research design.

---

# Why Stage 08 exists

Stage 07B showed:

- Logistic Regression underfitting / weak discrimination
- Random Forest currently strongest
- XGBoost not yet better than RF
- RF and XGBoost showing substantial train-test gaps
- noticeable performance variation between spatial folds

Simply making the models more complex may increase training performance while making spatial generalization worse.

Therefore this stage focuses on:

1. stronger static predictors;
2. modest feature engineering;
3. regularized model search;
4. threshold tuning inside training folds;
5. the exact same outer spatial validation design.

---

# What is added

## Existing numerical factors
- elevation
- slope
- aspect sin/cos
- clay
- sand
- silt
- bulk density
- distance to major fault

## Additional static/context factors
- JMG geological age
- JMG geological unit type
- current waterway distance, if local OSM waterway data are available

## Derived numerical factors
- log fault distance
- log waterway distance
- slope squared
- elevation × slope interaction

---

# What is NOT added here

The following stay in the later temporal/semi-dynamic experiment:

- rainfall
- ERA5 soil moisture
- historical land cover
- historical roads

Those variables change through time and should not be mixed blindly into a static historical susceptibility benchmark.

---

# Threshold tuning

ROC-AUC does not depend on a classification threshold.

However, recall, precision, F1 and balanced accuracy do.

Stage 07B used the default threshold:

`0.50`

This stage chooses a threshold **inside each outer training fold only**, using inner spatial out-of-fold predictions.

The threshold objective is:

> maximize balanced accuracy

while avoiding evaluation leakage.


In [1]:
# Optional:
# %pip install xgboost geopandas scikit-learn pandas numpy

from pathlib import Path
import math
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedGroupKFold,
    RandomizedSearchCV,
    cross_val_predict,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


pandas: 3.0.2


## 1. Project paths

In [2]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

INPUT_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "static_model_1to2.csv"
)

WATERWAYS_ROOT = (
    RAW_DIR
    / "03_Factors"
    / "Waterways"
)

OUTPUT_DIR = (
    DEV_DIR
    / "05_model_results"
    / "enhanced_static"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(INPUT_PATH)

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)


Input: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv
Output: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\enhanced_static


## 2. Load exactly the same Stage-06 samples

In [3]:
df = pd.read_csv(INPUT_PATH)

print("Rows:", len(df))
print("Landslides:", int((df["target"] == 1).sum()))
print("Pseudo-absences:", int((df["target"] == 0).sum()))
print("Spatial blocks:", df["spatial_block_id"].nunique())

if len(df) != 582:
    print(
        "NOTE: Stage 06 originally produced 582 rows. "
        "The notebook will still use the current file."
    )


Rows: 582
Landslides: 194
Pseudo-absences: 388
Spatial blocks: 167


# PART A — ADD CURRENT WATERWAY DISTANCE

## 3. Discover local waterway vectors

Waterways are treated as a relatively stable contextual factor.

This remains explicitly documented as current OSM waterway context rather than exact historical drainage reconstruction.


In [4]:
def discover_vectors(root):
    extensions = {
        ".shp",
        ".gpkg",
        ".geojson",
    }

    if not root.exists():
        return []

    return [
        p
        for p in root.rglob("*")
        if p.suffix.lower() in extensions
    ]


waterway_files = discover_vectors(
    WATERWAYS_ROOT
)

print("Waterway files found:", len(waterway_files))

for p in waterway_files[:20]:
    print(" -", p)


Waterway files found: 1
 - C:\Landslide_FYP\Landslide_FYP_Data\03_Factors\Waterways\gis_osm_waterways_free_1.shp


In [5]:
DISTANCE_CRS = "EPSG:3857"

if waterway_files:

    water_parts = []

    for path in waterway_files:

        try:
            gdf = gpd.read_file(path)

            if len(gdf) == 0 or gdf.crs is None:
                continue

            gdf = gdf.to_crs("EPSG:4326")

            gdf = gdf.loc[
                gdf.geometry.geom_type.isin(
                    [
                        "LineString",
                        "MultiLineString"
                    ]
                )
            ][["geometry"]].copy()

            if len(gdf):
                water_parts.append(gdf)

        except Exception as exc:
            print("Skipped:", path, "|", exc)


    if water_parts:

        waterways = gpd.GeoDataFrame(
            pd.concat(
                water_parts,
                ignore_index=True
            ),
            geometry="geometry",
            crs="EPSG:4326"
        ).to_crs(DISTANCE_CRS)

        water_union = waterways.geometry.unary_union

        sample_points = gpd.GeoDataFrame(
            df.copy(),
            geometry=gpd.points_from_xy(
                df["longitude"],
                df["latitude"]
            ),
            crs="EPSG:4326"
        ).to_crs(DISTANCE_CRS)

        df["distance_waterway_current_m"] = (
            sample_points.geometry.apply(
                lambda p: float(
                    p.distance(water_union)
                )
            ).to_numpy()
        )

        print(
            "Waterway distance available:",
            int(
                df[
                    "distance_waterway_current_m"
                ].notna().sum()
            ),
            "/",
            len(df)
        )

    else:
        df["distance_waterway_current_m"] = np.nan

else:
    df["distance_waterway_current_m"] = np.nan
    print("No local waterway vector found.")


Waterway distance available: 582 / 582


# PART B — FEATURE ENGINEERING

## 4. Numerical transformations

These transformations are deterministic and do not use the target label.


In [6]:
df["log_distance_major_fault_m"] = np.log1p(
    pd.to_numeric(
        df["distance_major_fault_m"],
        errors="coerce"
    )
)

df["slope_sq"] = (
    pd.to_numeric(
        df["slope_deg"],
        errors="coerce"
    ) ** 2
)

df["elevation_slope_interaction"] = (
    pd.to_numeric(
        df["elevation_m"],
        errors="coerce"
    )
    *
    pd.to_numeric(
        df["slope_deg"],
        errors="coerce"
    )
)

if df["distance_waterway_current_m"].notna().any():

    df["log_distance_waterway_current_m"] = np.log1p(
        pd.to_numeric(
            df["distance_waterway_current_m"],
            errors="coerce"
        )
    )

else:

    df["log_distance_waterway_current_m"] = np.nan


## 5. Predictor definitions

High-cardinality geology unit names are deliberately excluded from the first enhanced benchmark to reduce overfitting risk.

The lower-cardinality fields are:

- `geology_age`
- `geology_unit_type`

If either field is unavailable, the notebook automatically omits it.


In [7]:
NUMERIC_FEATURES = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
    "log_distance_major_fault_m",
    "slope_sq",
    "elevation_slope_interaction",
]

if (
    "distance_waterway_current_m"
    in df.columns
    and df[
        "distance_waterway_current_m"
    ].notna().any()
):
    NUMERIC_FEATURES.extend([
        "distance_waterway_current_m",
        "log_distance_waterway_current_m",
    ])


CATEGORICAL_FEATURES = [
    c
    for c in [
        "geology_age",
        "geology_unit_type",
    ]
    if c in df.columns
    and df[c].notna().any()
]


ALL_FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)


print("Numerical predictors:")
for c in NUMERIC_FEATURES:
    print(" -", c)

print("\nCategorical predictors:")
for c in CATEGORICAL_FEATURES:
    print(" -", c)


feature_availability = pd.DataFrame({
    "feature": ALL_FEATURES,
    "available": [
        int(df[c].notna().sum())
        for c in ALL_FEATURES
    ],
    "missing": [
        int(df[c].isna().sum())
        for c in ALL_FEATURES
    ],
})

display(feature_availability)

feature_availability.to_csv(
    OUTPUT_DIR / "enhanced_feature_availability.csv",
    index=False
)


Numerical predictors:
 - elevation_m
 - slope_deg
 - aspect_sin
 - aspect_cos
 - soil_clay
 - soil_sand
 - soil_silt
 - soil_bulk_density
 - distance_major_fault_m
 - log_distance_major_fault_m
 - slope_sq
 - elevation_slope_interaction
 - distance_waterway_current_m
 - log_distance_waterway_current_m

Categorical predictors:
 - geology_age
 - geology_unit_type


,feature,available,missing
0,elevation_m,582,0
1,slope_deg,582,0
2,aspect_sin,582,0
3,aspect_cos,582,0
4,soil_clay,582,0
5,soil_sand,582,0
6,soil_silt,582,0
7,soil_bulk_density,582,0
8,distance_major_fault_m,582,0
9,log_distance_major_fault_m,582,0


# PART C — IDENTICAL SPATIAL OUTER FOLDS

## 6. Prepare X, y and spatial groups


In [8]:
X = df[ALL_FEATURES].copy()
y = df["target"].astype(int).copy()
groups = df["spatial_block_id"].astype(str).copy()

RANDOM_SEED = 42
OUTER_SPLITS = 5
INNER_SPLITS = 3

outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

outer_folds = list(
    outer_splitter.split(
        X,
        y,
        groups
    )
)

print("Outer folds:", len(outer_folds))


Outer folds: 5


# PART D — LEAKAGE-SAFE PREPROCESSING

## 7. Preprocessors

For LR:
- median-impute numerical predictors
- standardize numerical predictors
- mode-impute categorical predictors
- one-hot encode categorical predictors

For RF/XGBoost:
- same imputation and one-hot encoding
- no numerical standardization required

All preprocessing is fitted only on training folds.


In [9]:
numeric_lr = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
])

numeric_tree = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
])

categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    ),
])


lr_preprocess = ColumnTransformer([
    (
        "num",
        numeric_lr,
        NUMERIC_FEATURES
    ),
    (
        "cat",
        categorical_pipe,
        CATEGORICAL_FEATURES
    ),
])


tree_preprocess = ColumnTransformer([
    (
        "num",
        numeric_tree,
        NUMERIC_FEATURES
    ),
    (
        "cat",
        categorical_pipe,
        CATEGORICAL_FEATURES
    ),
])


# PART E — MORE REGULARIZED MODEL SEARCH

## 8. Model definitions


In [10]:
lr_pipeline = Pipeline([
    (
        "preprocess",
        lr_preprocess
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_SEED
        )
    )
])


rf_pipeline = Pipeline([
    (
        "preprocess",
        tree_preprocess
    ),
    (
        "model",
        RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    )
])


xgb_pipeline = Pipeline([
    (
        "preprocess",
        tree_preprocess
    ),
    (
        "model",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            tree_method="hist"
        )
    )
])


MODEL_SPECS = {
    "Logistic Regression": {
        "pipeline": lr_pipeline,
        "params": {
            "model__C": [
                0.001,
                0.003,
                0.01,
                0.03,
                0.1,
                0.3,
                1.0,
                3.0,
                10.0,
            ],
        },
        "n_iter": 9,
    },

    "Random Forest": {
        "pipeline": rf_pipeline,
        "params": {
            "model__n_estimators": [
                300,
                500,
                800,
            ],
            "model__max_depth": [
                3,
                5,
                7,
                9,
                None,
            ],
            "model__min_samples_split": [
                5,
                10,
                20,
                30,
            ],
            "model__min_samples_leaf": [
                2,
                4,
                6,
                10,
                15,
            ],
            "model__max_features": [
                "sqrt",
                "log2",
                0.5,
            ],
        },
        "n_iter": 30,
    },

    "XGBoost": {
        "pipeline": xgb_pipeline,
        "params": {
            "model__n_estimators": [
                100,
                150,
                250,
                400,
            ],
            "model__max_depth": [
                2,
                3,
                4,
            ],
            "model__learning_rate": [
                0.01,
                0.03,
                0.05,
                0.1,
            ],
            "model__subsample": [
                0.6,
                0.75,
                0.9,
            ],
            "model__colsample_bytree": [
                0.6,
                0.75,
                0.9,
            ],
            "model__min_child_weight": [
                3,
                5,
                8,
                12,
            ],
            "model__reg_alpha": [
                0.0,
                0.1,
                0.5,
                1.0,
            ],
            "model__reg_lambda": [
                2.0,
                5.0,
                10.0,
                20.0,
            ],
            "model__scale_pos_weight": [
                1.0,
                1.5,
                2.0,
            ],
        },
        "n_iter": 35,
    },
}


# PART F — THRESHOLD OPTIMIZATION

## 9. Select threshold from inner-training predictions only


In [11]:
def metric_dict(
    y_true,
    prob,
    threshold
):

    pred = (
        np.asarray(prob)
        >= threshold
    ).astype(int)

    return {
        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                prob
            ),
    }


def choose_balanced_accuracy_threshold(
    y_true,
    prob
):

    thresholds = np.linspace(
        0.15,
        0.85,
        141
    )

    rows = []

    for threshold in thresholds:

        metrics = metric_dict(
            y_true,
            prob,
            threshold
        )

        rows.append({
            "threshold":
                float(
                    threshold
                ),

            "balanced_accuracy":
                metrics[
                    "balanced_accuracy"
                ],

            "recall":
                metrics[
                    "recall"
                ],

            "precision":
                metrics[
                    "precision"
                ],

            "f1":
                metrics[
                    "f1"
                ],
        })

    table = pd.DataFrame(
        rows
    )

    # Primary objective: max balanced accuracy.
    # Tie-breaker: greater recall, then F1.
    best = table.sort_values(
        [
            "balanced_accuracy",
            "recall",
            "f1",
        ],
        ascending=False
    ).iloc[0]

    return (
        float(
            best[
                "threshold"
            ]
        ),
        table
    )


# PART G — NESTED SPATIAL TRAINING

## 10. Tune hyperparameters + threshold without touching outer test folds


In [12]:
METRICS = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]

results = []
parameter_rows = []
threshold_rows = []
oof_rows = []

for model_name, spec in MODEL_SPECS.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    for fold, (train_idx, test_idx) in enumerate(
        outer_folds,
        start=1
    ):

        X_train = X.iloc[train_idx].copy()
        y_train = y.iloc[train_idx].copy()
        g_train = groups.iloc[train_idx].copy()

        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        inner = StratifiedGroupKFold(
            n_splits=INNER_SPLITS,
            shuffle=True,
            random_state=RANDOM_SEED + fold
        )

        inner_splits = list(
            inner.split(
                X_train,
                y_train,
                g_train
            )
        )

        search = RandomizedSearchCV(
            estimator=clone(
                spec["pipeline"]
            ),
            param_distributions=spec[
                "params"
            ],
            n_iter=spec[
                "n_iter"
            ],
            scoring="roc_auc",
            cv=inner_splits,
            refit=True,
            random_state=RANDOM_SEED + fold,
            n_jobs=-1,
            verbose=0,
        )

        start = time.time()

        search.fit(
            X_train,
            y_train
        )

        best_model = search.best_estimator_

        # Inner OOF predictions from the chosen model.
        # Used ONLY to choose classification threshold.
        inner_oof_prob = cross_val_predict(
            clone(best_model),
            X_train,
            y_train,
            cv=inner_splits,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        best_threshold, threshold_table = (
            choose_balanced_accuracy_threshold(
                y_train,
                inner_oof_prob
            )
        )

        # Refit chosen pipeline on the complete outer-training set.
        best_model.fit(
            X_train,
            y_train
        )

        train_prob = best_model.predict_proba(
            X_train
        )[:, 1]

        test_prob = best_model.predict_proba(
            X_test
        )[:, 1]

        train_metrics = metric_dict(
            y_train,
            train_prob,
            best_threshold
        )

        test_metrics = metric_dict(
            y_test,
            test_prob,
            best_threshold
        )

        # Also retain default 0.50 results.
        test_default = metric_dict(
            y_test,
            test_prob,
            0.5
        )

        row = {
            "model":
                model_name,

            "outer_fold":
                fold,

            "best_threshold":
                best_threshold,

            "inner_best_roc_auc":
                search.best_score_,

            "fit_seconds":
                time.time() - start,
        }

        for metric in METRICS:

            row[
                f"train_{metric}"
            ] = train_metrics[
                metric
            ]

            row[
                f"test_{metric}"
            ] = test_metrics[
                metric
            ]

            row[
                f"gap_{metric}"
            ] = (
                train_metrics[
                    metric
                ]
                -
                test_metrics[
                    metric
                ]
            )

            row[
                f"test_default05_{metric}"
            ] = test_default[
                metric
            ]

        results.append(row)

        parameter_rows.append({
            "model":
                model_name,

            "outer_fold":
                fold,

            "best_params":
                str(
                    search.best_params_
                ),
        })

        threshold_rows.append({
            "model":
                model_name,

            "outer_fold":
                fold,

            "selected_threshold":
                best_threshold,

            "inner_oof_balanced_accuracy":
                metric_dict(
                    y_train,
                    inner_oof_prob,
                    best_threshold
                )[
                    "balanced_accuracy"
                ],

            "inner_oof_recall":
                metric_dict(
                    y_train,
                    inner_oof_prob,
                    best_threshold
                )[
                    "recall"
                ],

            "inner_oof_f1":
                metric_dict(
                    y_train,
                    inner_oof_prob,
                    best_threshold
                )[
                    "f1"
                ],
        })

        test_pred = (
            test_prob
            >= best_threshold
        ).astype(int)

        for pos, original_idx in enumerate(
            test_idx
        ):

            oof_rows.append({
                "model":
                    model_name,

                "outer_fold":
                    fold,

                "row_index":
                    int(
                        original_idx
                    ),

                "sample_id":
                    df.iloc[
                        original_idx
                    ][
                        "sample_id"
                    ],

                "target":
                    int(
                        y.iloc[
                            original_idx
                        ]
                    ),

                "probability":
                    float(
                        test_prob[
                            pos
                        ]
                    ),

                "threshold":
                    best_threshold,

                "prediction":
                    int(
                        test_pred[
                            pos
                        ]
                    ),

                "spatial_block_id":
                    groups.iloc[
                        original_idx
                    ],
            })

        print(
            "Fold",
            fold,
            "| threshold:",
            round(
                best_threshold,
                3
            ),
            "| Test AUC:",
            round(
                test_metrics[
                    "roc_auc"
                ],
                3
            ),
            "| BA:",
            round(
                test_metrics[
                    "balanced_accuracy"
                ],
                3
            ),
            "| Recall:",
            round(
                test_metrics[
                    "recall"
                ],
                3
            )
        )


fold_results = pd.DataFrame(
    results
)

best_params = pd.DataFrame(
    parameter_rows
)

threshold_summary = pd.DataFrame(
    threshold_rows
)

oof = pd.DataFrame(
    oof_rows
)

fold_results.to_csv(
    OUTPUT_DIR
    / "enhanced_nested_fold_results.csv",
    index=False
)

best_params.to_csv(
    OUTPUT_DIR
    / "enhanced_best_hyperparameters.csv",
    index=False
)

threshold_summary.to_csv(
    OUTPUT_DIR
    / "enhanced_thresholds_by_fold.csv",
    index=False
)

oof.to_csv(
    OUTPUT_DIR
    / "enhanced_oof_predictions.csv",
    index=False
)

display(
    fold_results
)



Logistic Regression
Fold 1 | threshold: 0.24 | Test AUC: 0.934 | BA: 0.751 | Recall: 0.925
Fold 2 | threshold: 0.645 | Test AUC: 0.821 | BA: 0.618 | Recall: 0.4
Fold 3 | threshold: 0.26 | Test AUC: 0.857 | BA: 0.785 | Recall: 0.947
Fold 4 | threshold: 0.6 | Test AUC: 0.855 | BA: 0.771 | Recall: 0.658
Fold 5 | threshold: 0.425 | Test AUC: 0.908 | BA: 0.81 | Recall: 0.816

Random Forest
Fold 1 | threshold: 0.49 | Test AUC: 0.963 | BA: 0.905 | Recall: 0.925
Fold 2 | threshold: 0.4 | Test AUC: 0.86 | BA: 0.867 | Recall: 0.975
Fold 3 | threshold: 0.49 | Test AUC: 0.901 | BA: 0.784 | Recall: 0.684
Fold 4 | threshold: 0.5 | Test AUC: 0.869 | BA: 0.771 | Recall: 0.737
Fold 5 | threshold: 0.435 | Test AUC: 0.933 | BA: 0.849 | Recall: 0.789

XGBoost
Fold 1 | threshold: 0.595 | Test AUC: 0.954 | BA: 0.861 | Recall: 0.825
Fold 2 | threshold: 0.275 | Test AUC: 0.86 | BA: 0.855 | Recall: 0.975
Fold 3 | threshold: 0.57 | Test AUC: 0.896 | BA: 0.797 | Recall: 0.684
Fold 4 | threshold: 0.45 | Test AUC

,model,outer_fold,best_threshold,inner_best_roc_auc,fit_seconds,train_accuracy,test_accuracy,gap_accuracy,test_default05_accuracy,train_balanced_accuracy,...,gap_f1,test_default05_f1,train_roc_auc,test_roc_auc,gap_roc_auc,test_default05_roc_auc,train_pr_auc,test_pr_auc,gap_pr_auc,test_default05_pr_auc
0,Logistic Regression,1,0.240,0.872667,20.041090,0.767241,0.694915,0.072326,0.838983,0.816003,...,0.059946,0.791209,0.926393,0.934295,-0.007902,0.934295,0.851209,0.881372,-0.030163,0.881372
1,Logistic Regression,2,0.645,0.847530,10.317391,0.866091,0.689076,0.177015,0.823529,0.845932,...,0.332285,0.774194,0.937503,0.820570,0.116933,0.820570,0.885222,0.553556,0.331665,0.553556
2,Logistic Regression,3,0.260,0.851904,9.402598,0.762313,0.730435,0.031878,0.800000,0.810362,...,0.029577,0.741573,0.938742,0.857143,0.081599,0.857143,0.879035,0.675106,0.203929,0.675106
3,Logistic Regression,4,0.600,0.908816,10.299067,0.873662,0.808696,0.064966,0.782609,0.860417,...,0.118254,0.666667,0.941463,0.855092,0.086370,0.855092,0.889802,0.775073,0.114730,0.775073
4,Logistic Regression,5,0.425,0.859957,10.969305,0.826552,0.808696,0.017857,0.843478,0.850606,...,0.042393,0.763158,0.936145,0.908066,0.028079,0.908066,0.885161,0.743454,0.141707,0.743454
5,Random Forest,1,0.490,0.863237,105.631159,0.875000,0.898305,-0.023305,0.898305,0.883578,...,-0.032063,0.860465,0.959677,0.963462,-0.003784,0.963462,0.932071,0.942496,-0.010425,0.942496
6,Random Forest,2,0.400,0.867884,90.157666,0.928726,0.831933,0.096793,0.798319,0.946602,...,0.107307,0.720930,0.996196,0.860443,0.135753,0.860443,0.992511,0.678349,0.314162,0.678349
7,Random Forest,3,0.490,0.888727,101.982174,0.890792,0.817391,0.073401,0.817391,0.894045,...,0.134518,0.712329,0.965104,0.900547,0.064557,0.900547,0.938039,0.786805,0.151235,0.786805
8,Random Forest,4,0.500,0.915427,92.240750,0.950749,0.782609,0.168141,0.782609,0.955035,...,0.237873,0.691358,0.992765,0.869105,0.123661,0.869105,0.987027,0.769119,0.217908,0.769119
9,Random Forest,5,0.435,0.883342,68.085658,0.942184,0.869565,0.072619,0.852174,0.953397,...,0.119403,0.767123,0.995053,0.933356,0.061697,0.933356,0.990869,0.856608,0.134261,0.856608


# PART H — FINAL ENHANCED RESULTS

## 11. Mean ± standard deviation


In [13]:
summary_rows = []

for model_name, group in fold_results.groupby(
    "model"
):

    row = {
        "model":
            model_name,

        "mean_selected_threshold":
            group[
                "best_threshold"
            ].mean(),
    }

    for metric in METRICS:

        row[
            f"{metric}_mean"
        ] = group[
            f"test_{metric}"
        ].mean()

        row[
            f"{metric}_std"
        ] = group[
            f"test_{metric}"
        ].std()

    row[
        "train_roc_auc_mean"
    ] = group[
        "train_roc_auc"
    ].mean()

    row[
        "test_roc_auc_mean"
    ] = group[
        "test_roc_auc"
    ].mean()

    row[
        "roc_auc_gap_mean"
    ] = group[
        "gap_roc_auc"
    ].mean()

    row[
        "train_f1_mean"
    ] = group[
        "train_f1"
    ].mean()

    row[
        "test_f1_mean"
    ] = group[
        "test_f1"
    ].mean()

    row[
        "f1_gap_mean"
    ] = group[
        "gap_f1"
    ].mean()

    summary_rows.append(row)


enhanced_summary = pd.DataFrame(
    summary_rows
).sort_values(
    "roc_auc_mean",
    ascending=False
)

display(
    enhanced_summary
)

enhanced_summary.to_csv(
    OUTPUT_DIR
    / "enhanced_model_summary.csv",
    index=False
)


,model,mean_selected_threshold,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,...,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,train_roc_auc_mean,test_roc_auc_mean,roc_auc_gap_mean,train_f1_mean,test_f1_mean,f1_gap_mean
1,Random Forest,0.463,0.839961,0.045121,0.835204,0.056670,0.736318,0.073408,0.822105,0.123808,...,0.905382,0.043344,0.806675,0.099012,0.981759,0.905382,0.076377,0.885422,0.772014,0.113408
2,XGBoost,0.467,0.833254,0.041052,0.829252,0.042458,0.730736,0.083391,0.817895,0.107455,...,0.903359,0.038652,0.804187,0.100866,0.960846,0.903359,0.057486,0.841218,0.765562,0.075656
0,Logistic Regression,0.434,0.746363,0.059062,0.747011,0.075473,0.608670,0.090767,0.749211,0.226323,...,0.875033,0.045517,0.725712,0.121635,0.936049,0.875033,0.061016,0.770104,0.653613,0.116491


## 12. Overall held-out metrics and confusion matrices


In [14]:
overall_rows = []
confusion_rows = []

for model_name, group in oof.groupby(
    "model"
):

    group = group.sort_values(
        "row_index"
    )

    y_true = group[
        "target"
    ].to_numpy()

    prob = group[
        "probability"
    ].to_numpy()

    pred = group[
        "prediction"
    ].to_numpy()

    overall_rows.append({
        "model":
            model_name,

        "accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                pred
            ),

        "precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                prob
            ),
    })

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred
    ).ravel()

    confusion_rows.append({
        "model":
            model_name,

        "true_negative":
            int(tn),

        "false_positive":
            int(fp),

        "false_negative":
            int(fn),

        "true_positive":
            int(tp),
    })


overall_metrics = pd.DataFrame(
    overall_rows
).sort_values(
    "roc_auc",
    ascending=False
)

confusion = pd.DataFrame(
    confusion_rows
)

display(
    overall_metrics
)

display(
    confusion
)

overall_metrics.to_csv(
    OUTPUT_DIR
    / "enhanced_model_overall_metrics.csv",
    index=False
)

confusion.to_csv(
    OUTPUT_DIR
    / "enhanced_confusion_matrices.csv",
    index=False
)


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
1,Random Forest,0.840206,0.836340,0.730594,0.824742,0.774818,0.907031,0.797490
2,XGBoost,0.833333,0.829897,0.719457,0.819588,0.766265,0.890776,0.780658
0,Logistic Regression,0.745704,0.746134,0.594262,0.747423,0.662100,0.879969,0.745002


,model,true_negative,false_positive,false_negative,true_positive
0,Logistic Regression,289,99,49,145
1,Random Forest,329,59,34,160
2,XGBoost,326,62,35,159


# PART I — COMPARE AGAINST STAGE 07B

## 13. Automatic baseline-vs-enhanced comparison

If the previous Stage-07B summary exists, this cell compares the new results directly.


In [15]:
BASELINE_SUMMARY_PATH = (
    DEV_DIR
    / "05_model_results"
    / "final_LR_RF_XGB_benchmark"
    / "final_model_benchmark_summary.csv"
)

if BASELINE_SUMMARY_PATH.exists():

    baseline = pd.read_csv(
        BASELINE_SUMMARY_PATH
    )

    comparison = baseline[
        [
            "model",
            "roc_auc_mean",
            "balanced_accuracy_mean",
            "recall_mean",
            "f1_mean",
            "pr_auc_mean",
            "roc_auc_overfit_gap_mean",
        ]
    ].rename(
        columns={
            "roc_auc_mean":
                "baseline_roc_auc",

            "balanced_accuracy_mean":
                "baseline_balanced_accuracy",

            "recall_mean":
                "baseline_recall",

            "f1_mean":
                "baseline_f1",

            "pr_auc_mean":
                "baseline_pr_auc",

            "roc_auc_overfit_gap_mean":
                "baseline_roc_auc_gap",
        }
    )

    enhanced_compare = enhanced_summary[
        [
            "model",
            "roc_auc_mean",
            "balanced_accuracy_mean",
            "recall_mean",
            "f1_mean",
            "pr_auc_mean",
            "roc_auc_gap_mean",
        ]
    ].rename(
        columns={
            "roc_auc_mean":
                "enhanced_roc_auc",

            "balanced_accuracy_mean":
                "enhanced_balanced_accuracy",

            "recall_mean":
                "enhanced_recall",

            "f1_mean":
                "enhanced_f1",

            "pr_auc_mean":
                "enhanced_pr_auc",

            "roc_auc_gap_mean":
                "enhanced_roc_auc_gap",
        }
    )

    comparison = comparison.merge(
        enhanced_compare,
        on="model",
        how="outer"
    )

    for metric in [
        "roc_auc",
        "balanced_accuracy",
        "recall",
        "f1",
        "pr_auc",
    ]:

        comparison[
            f"change_{metric}"
        ] = (
            comparison[
                f"enhanced_{metric}"
            ]
            -
            comparison[
                f"baseline_{metric}"
            ]
        )

    display(
        comparison
    )

    comparison.to_csv(
        OUTPUT_DIR
        / "baseline_vs_enhanced_comparison.csv",
        index=False
    )

else:

    print(
        "Previous Stage-07B summary not found. "
        "Enhanced outputs were still created normally."
    )


,model,baseline_roc_auc,baseline_balanced_accuracy,baseline_recall,baseline_f1,baseline_pr_auc,baseline_roc_auc_gap,enhanced_roc_auc,enhanced_balanced_accuracy,enhanced_recall,enhanced_f1,enhanced_pr_auc,enhanced_roc_auc_gap,change_roc_auc,change_balanced_accuracy,change_recall,change_f1,change_pr_auc
0,Logistic Regression,0.558682,0.542894,0.459474,0.404694,0.446116,0.138328,0.875033,0.747011,0.749211,0.653613,0.725712,0.061016,0.316351,0.204117,0.289737,0.248919,0.279597
1,Random Forest,0.706103,0.648944,0.470263,0.511291,0.594019,0.264429,0.905382,0.835204,0.822105,0.772014,0.806675,0.076377,0.199280,0.186260,0.351842,0.260723,0.212656
2,XGBoost,0.709606,0.626802,0.351842,0.409584,0.591218,0.263683,0.903359,0.829252,0.817895,0.765562,0.804187,0.057486,0.193754,0.202449,0.466053,0.355978,0.212969


# How to interpret Stage 08

A genuine improvement should ideally show some combination of:

- higher outer-test ROC-AUC;
- higher PR-AUC;
- higher balanced accuracy;
- substantially better landslide recall;
- higher F1;
- smaller fold standard deviation;
- reduced train-test gap.

Do **not** call the model improved merely because:

- training ROC-AUC increases;
- training accuracy increases;
- accuracy increases while recall collapses.

---

# What each modelling stage has trained so far

## Stage 04
No ML model.

Prepared reliable point-level landslide cases and background candidates.

## Stage 05A
No ML model.

Attached static conditioning factors to the samples.

## Stage 05B
No final classifier.

Constructed time-matched data for future dynamic experiments.

## Stage 06
No classifier.

Selected the final 1:2 pseudo-absence set and spatial validation groups.

## Stage 07
First simple LR and RF static baseline.

Purpose:
check whether the basic static factors contain useful landslide-susceptibility signal.

## Stage 07B
Formal LR vs RF vs XGBoost benchmark.

Purpose:
compare algorithms fairly using nested spatial validation and hyperparameter tuning.

## Stage 08
Enhanced static benchmark.

Purpose:
test whether adding defensible geology/context information, engineered features, stronger regularization and leakage-safe threshold tuning improves generalization.

---

# What comes after Stage 08

If Stage 08 improves performance:

### Stage 09 — Temporal / semi-dynamic model comparison

Use the already-created temporal subsets to test whether:

- rainfall
- soil moisture
- historical land cover

provide additional predictive value beyond static susceptibility.

Historical road distance can also be tested as an optional factor if ohsome coverage is adequate.

The final dashboard can then separate:

1. **baseline susceptibility**
2. **semi-dynamic condition / hazard indication**
3. **exposure overlays such as roads**


# Outputs to send for review

Please send:

```text
enhanced_feature_availability.csv
enhanced_model_summary.csv
enhanced_model_overall_metrics.csv
enhanced_confusion_matrices.csv
enhanced_thresholds_by_fold.csv
baseline_vs_enhanced_comparison.csv
```

If `baseline_vs_enhanced_comparison.csv` is not created, send the first five files and we can compare them manually.
